# Experiments

Dieses Notebook enth?lt die systematische Fidelity/Sparsity-Evaluation f?r GNNExplainer und Integrated Gradients.

- Fidelity wird getrennt f?r `H` und `C` aggregiert.
- Standardm??ig wird nur der **erste Graph pro `compound`** im gew?hlten Scope verwendet.
- Scope kann zwischen `test_split` und `full_dataset` umgeschaltet werden.


In [2]:
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output


def _resolve_project_root() -> Path:
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "gnn4nmr",
        cwd.parent / "gnn4nmr",
    ]
    for candidate in candidates:
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError(
        f"Could not resolve project root from cwd={cwd}. Expected a folder containing scripts/ and notebooks/."
    )


PROJECT_ROOT = _resolve_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [str(PROJECT_ROOT), str(SCRIPTS_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"cwd: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Models dir: {MODELS_DIR}")


cwd: /Users/sophiaberg/gnn4nmr-7/notebooks
Project root: /Users/sophiaberg/gnn4nmr-7
Notebook dir: /Users/sophiaberg/gnn4nmr-7/notebooks
Data dir: /Users/sophiaberg/gnn4nmr-7/data
Models dir: /Users/sophiaberg/gnn4nmr-7/models


In [3]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

# Make this cell runnable on its own (even after kernel restart)
if "NOTEBOOK_DIR" not in globals():
    NOTEBOOK_DIR = Path.cwd()
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
if "MODELS_DIR" not in globals():
    MODELS_DIR = PROJECT_ROOT / "models"
if "DATA_DIR" not in globals():
    DATA_DIR = PROJECT_ROOT / "data"


def _list_files(directory: Path, extension: str):
    if directory.exists():
        return sorted([f.name for f in directory.glob(f"*{extension}")])
    return []


model_files = _list_files(MODELS_DIR, ".pt")
data_files = _list_files(DATA_DIR, ".pkl")

exp_model_widget = widgets.Dropdown(
    options=model_files if model_files else ["Keine Modelle gefunden"],
    description="Modell:",
    style={"description_width": "initial"},
)

exp_data_widget = widgets.Dropdown(
    options=data_files if data_files else ["Keine Daten gefunden"],
    description="Daten:",
    style={"description_width": "initial"},
)

split_file_widget = widgets.Text(
    value="models/graph_split.pkl",
    description="Split Datei:",
    style={"description_width": "initial"},
)

graph_scope_widget = widgets.Dropdown(
    options=[("Testsplit", "test_split"), ("Gesamtdatensatz", "full_dataset")],
    value="test_split",
    description="Graph Scope:",
    style={"description_width": "initial"},
)

first_graph_per_component_widget = widgets.Checkbox(
    value=True,
    description="Erster Graph pro Compound",
    indent=False,
)

component_key_widget = widgets.Text(
    value="compound",
    description="Component Key:",
    style={"description_width": "initial"},
)

node_types_widget = widgets.SelectMultiple(
    options=["H", "C", "Others"],
    value=("H", "C"),
    description="Node Types:",
    style={"description_width": "initial"},
)

sparsity_widget = widgets.FloatSlider(
    value=0.90,
    min=0.50,
    max=0.99,
    step=0.01,
    description="Sparsity:",
    style={"description_width": "initial"},
    readout_format=".2f",
)

mask_baseline_widget = widgets.Dropdown(
    options=["match_ig_baseline", "zero", "mean"],
    value="match_ig_baseline",
    description="Mask Baseline:",
    style={"description_width": "initial"},
)

ig_baseline_widget = widgets.Dropdown(
    options=["scientific", "zero", "mean", "random", "min", "max"],
    value="scientific",
    description="IG Baseline:",
    style={"description_width": "initial"},
)

gnn_epochs_widget = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description="GNN Epochs:",
    style={"description_width": "initial"},
)

gnn_lr_widget = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-4,
    max=-1,
    step=0.1,
    description="GNN LR:",
    style={"description_width": "initial"},
    readout_format=".4f",
)

gnn_explanation_type_widget = widgets.Dropdown(
    options=["phenomenon", "model"],
    value="phenomenon",
    description="GNN Type:",
    style={"description_width": "initial"},
)

ig_n_steps_widget = widgets.IntSlider(
    value=64,
    min=8,
    max=256,
    step=8,
    description="IG Steps:",
    style={"description_width": "initial"},
)

max_graphs_widget = widgets.IntText(
    value=0,
    description="Max Graphen (0=all):",
    style={"description_width": "initial"},
)

max_nodes_per_graph_widget = widgets.IntText(
    value=0,
    description="Max Nodes/Graph (0=all):",
    style={"description_width": "initial"},
)

include_edge_report_widget = widgets.Checkbox(
    value=True,
    description="GNN Edge Report",
    indent=False,
)

seed_widget = widgets.IntText(
    value=0,
    description="Seed:",
    style={"description_width": "initial"},
)

output_dir_widget = widgets.Text(
    value="results/experiments",
    description="Output Dir:",
    style={"description_width": "initial"},
)

controls = widgets.VBox(
    [
        widgets.HBox([exp_model_widget, exp_data_widget]),
        widgets.HBox([split_file_widget, graph_scope_widget]),
        widgets.HBox([first_graph_per_component_widget, component_key_widget]),
        widgets.HBox([node_types_widget, sparsity_widget]),
        widgets.HBox([mask_baseline_widget, ig_baseline_widget]),
        widgets.HBox([gnn_epochs_widget, gnn_lr_widget, gnn_explanation_type_widget]),
        widgets.HBox([ig_n_steps_widget, max_graphs_widget, max_nodes_per_graph_widget]),
        widgets.HBox([include_edge_report_widget, seed_widget, output_dir_widget]),
    ]
)

display(controls)


## Fidelity and Sparsity Metrics (Node-Level Regression)

Fuer jeden Knoten und jede Methode wird bei Ziel-Sparsity `s=0.90` zunaechst `k = max(1, ceil((1-s) * d))` (mit `d` = Feature-Anzahl) bestimmt.

- `S`: Top-k Features nach `|importance|`
- `actual_sparsity_feat = 1 - k/d`

Feature-Perturbationen am ausgewaehlten Knoten:

- `x_drop`: Features in `S` werden durch Baseline ersetzt
- `x_keep`: nur Features in `S` bleiben, Rest wird durch Baseline ersetzt

Mit `y_orig`, `y_drop`, `y_keep` und optional `y_true`:

- `fid_plus_model = |y_orig - y_drop|`
- `fid_minus_model = |y_orig - y_keep|`
- `fid_plus_error_delta = |y_drop - y_true| - |y_orig - y_true|`
- `fid_minus_error_delta = |y_keep - y_true| - |y_orig - y_true|`

Fuer den optionalen GNN-Edge-Report wird die Edge-Selektion global auf dem gesamten erklaerten Graphen durchgefuehrt:

- `E`: Anzahl aller beruecksichtigten Edges im Graphen
- `k_edge = max(1, ceil((1-s) * E))`, Ranking nach `|edge_mask|`
- Edge-`fid+`: Top-k Edges werden entfernt (`drop_selected`)
- Edge-`fid-`: Es bleiben nur die Top-k Edges erhalten (`keep_selected`)
- `actual_sparsity_edge = 1 - k_edge/E`

Die Aggregation erfolgt getrennt fuer `H` und `C`. Der faire Methodenvergleich nutzt nur die gemeinsame Node-Menge von GNNExplainer und IG.



In [4]:
from scripts.explainer.experiments_evaluation import (
    exp_load_eval_graph_indices,
    exp_build_mask_baseline,
    exp_topk_indices_from_importance,
    exp_predict_single_node,
    exp_feature_fidelity_for_node,
    exp_gnn_edge_fidelity_for_node,
    exp_extract_gnn_feature_importance,
    exp_extract_ig_feature_importance,
    select_scope_graph_indices,
    first_graph_indices_per_component,
    build_default_context,
    run_experiments_evaluation as _run_experiments_evaluation_core,
)


def run_experiments_evaluation(
    model_file=None,
    data_file=None,
    split_file=None,
    node_types=('H', 'C'),
    sparsity=0.90,
    mask_baseline_mode='match_ig_baseline',
    ig_baseline_mode='scientific',
    gnn_epochs=200,
    gnn_lr=0.01,
    gnn_explanation_type='phenomenon',
    ig_n_steps=64,
    graph_scope='test_split',
    first_graph_per_component=True,
    component_key='compound',
    max_graphs=None,
    max_nodes_per_graph=0,
    include_gnn_edge_report=True,
    output_dir='results/experiments',
    seed=0,
    verbose=True,
):
    model_name = model_file if model_file is not None else exp_model_widget.value
    data_name = data_file if data_file is not None else exp_data_widget.value

    if str(model_name).startswith('Keine') or str(data_name).startswith('Keine'):
        raise ValueError('Bitte g?ltige Modell- und Datendatei ausw?hlen.')

    split_eff = split_file if split_file is not None else split_file_widget.value

    context = build_default_context(
        project_root=PROJECT_ROOT,
        model_file=str(model_name),
        data_file=str(data_name),
        split_file=str(split_eff),
        output_dir=output_dir,
    )

    return _run_experiments_evaluation_core(
        context=context,
        node_types=node_types,
        sparsity=float(sparsity),
        mask_baseline_mode=str(mask_baseline_mode),
        ig_baseline_mode=str(ig_baseline_mode),
        gnn_epochs=int(gnn_epochs),
        gnn_lr=float(gnn_lr),
        gnn_explanation_type=str(gnn_explanation_type),
        gnn_use_custom_coeffs=False,
        gnn_coeffs=None,
        ig_n_steps=int(ig_n_steps),
        graph_scope=str(graph_scope),
        first_graph_per_component=bool(first_graph_per_component),
        component_key=str(component_key),
        max_graphs=max_graphs,
        max_nodes_per_graph=int(max_nodes_per_graph),
        include_gnn_edge_report=bool(include_gnn_edge_report),
        seed=int(seed),
        verbose=bool(verbose),
    )


In [5]:
run_button = widgets.Button(
    description='Experiments ausf?hren',
    button_style='success',
    icon='play'
)
run_output = widgets.Output()


def _run_clicked(_):
    with run_output:
        clear_output()
        try:
            global exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df
            max_graphs = int(max_graphs_widget.value)
            max_graphs = None if max_graphs <= 0 else max_graphs

            exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df = run_experiments_evaluation(
                node_types=tuple(node_types_widget.value),
                sparsity=float(sparsity_widget.value),
                mask_baseline_mode=mask_baseline_widget.value,
                ig_baseline_mode=ig_baseline_widget.value,
                gnn_epochs=int(gnn_epochs_widget.value),
                gnn_lr=float(gnn_lr_widget.value),
                gnn_explanation_type=gnn_explanation_type_widget.value,
                ig_n_steps=int(ig_n_steps_widget.value),
                graph_scope=graph_scope_widget.value,
                first_graph_per_component=bool(first_graph_per_component_widget.value),
                component_key=component_key_widget.value.strip() or 'compound',
                max_graphs=max_graphs,
                max_nodes_per_graph=int(max_nodes_per_graph_widget.value),
                include_gnn_edge_report=bool(include_edge_report_widget.value),
                output_dir=output_dir_widget.value,
                seed=int(seed_widget.value),
                verbose=True,
            )

            print('Summary by method/node type:')
            display(exp_summary_method_type_df)
            print('Fair comparison (shared nodes):')
            display(exp_summary_fair_df)
        except Exception as exc:
            print(f'? Fehler: {exc}')
            raise


run_button.on_click(_run_clicked)
display(run_button)
display(run_output)


Button(button_style='success', description='Experiments ausf?hren', icon='play', style=ButtonStyle())

Output()

In [6]:
# Optionaler Smoke-Run (bewusst auskommentiert):
# exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df = run_experiments_evaluation(
#     node_types=('H', 'C'),
#     sparsity=0.90,
#     mask_baseline_mode='match_ig_baseline',
#     ig_baseline_mode='scientific',
#     graph_scope='test_split',
#     first_graph_per_component=True,
#     component_key='compound',
#     max_graphs=2,
#     max_nodes_per_graph=5,
#     include_gnn_edge_report=True,
#     seed=0,
#     verbose=True,
# )
# display(exp_summary_method_type_df)
# display(exp_summary_fair_df)
